# 5. Abstract Classes

Abstract classes define a **contract** that subclasses must fulfill — same concept as Java's `abstract class`. Python uses the `abc` module (Abstract Base Classes) to enforce this.

This notebook covers: `ABC`, `@abstractmethod`, abstract properties, abstract methods with a body, concrete methods in abstract classes, and when to use ABC vs Protocol.

### 5.1 Defining an Abstract Class

**☕ JAVA:**
```java
public abstract class Shape {
    public abstract double area();
    public abstract double perimeter();

    // Concrete method in abstract class
    public String describe() {
        return getClass().getSimpleName() + ": area=" + area();
    }
}
```

**🐍 PYTHON:** Inherit from `ABC` and use `@abstractmethod`. You can also have concrete (non-abstract) methods in the same class.

In [ ]:
from abc import ABC, abstractmethod
import math

class Shape(ABC):
    """Abstract base class for shapes."""

    @abstractmethod
    def area(self) -> float:
        """Subclasses MUST implement this."""
        ...

    @abstractmethod
    def perimeter(self) -> float:
        """Subclasses MUST implement this."""
        ...

    def describe(self) -> str:
        """Concrete method — shared by all subclasses."""
        return f"{self.__class__.__name__}: area={self.area():.2f}, perimeter={self.perimeter():.2f}"

In [ ]:
# Can't instantiate abstract class!
try:
    s = Shape()
except TypeError as e:
    print(f"❌ {e}")

### 5.2 Implementing an Abstract Class

**☕ JAVA:** Subclass uses `extends` and must implement all abstract methods, or be abstract itself.

**🐍 PYTHON:** Same rule — all `@abstractmethod` methods must be implemented, or `TypeError` at instantiation.

In [ ]:
class Rectangle(Shape):
    def __init__(self, width: float, height: float):
        self.width = width
        self.height = height

    def area(self) -> float:
        return self.width * self.height

    def perimeter(self) -> float:
        return 2 * (self.width + self.height)

class Circle(Shape):
    def __init__(self, radius: float):
        self.radius = radius

    def area(self) -> float:
        return math.pi * self.radius ** 2

    def perimeter(self) -> float:
        return 2 * math.pi * self.radius

# Polymorphism — same as Java
shapes: list[Shape] = [Rectangle(10, 5), Circle(7)]
for shape in shapes:
    print(f"  {shape.describe()}")   # Uses concrete method from ABC

In [ ]:
# What happens if you miss an abstract method?
class IncompleteShape(Shape):
    def area(self) -> float:
        return 0.0
    # Missing perimeter()!

try:
    s = IncompleteShape()
except TypeError as e:
    print(f"❌ {e}")

### 5.3 Abstract Properties

**☕ JAVA:** Abstract getters are just abstract methods: `public abstract String getName();`

**🐍 PYTHON:** Combine `@property` with `@abstractmethod` — the subclass must implement it as a property.

In [ ]:
class Animal(ABC):
    @property
    @abstractmethod
    def sound(self) -> str:
        """Each animal must define its sound."""
        ...

    @property
    @abstractmethod
    def legs(self) -> int:
        """Each animal must define its number of legs."""
        ...

    def describe(self) -> str:
        return f"{self.__class__.__name__}: {self.sound}, {self.legs} legs"

class Dog(Animal):
    @property
    def sound(self) -> str:
        return "Woof!"

    @property
    def legs(self) -> int:
        return 4

class Snake(Animal):
    @property
    def sound(self) -> str:
        return "Hiss!"

    @property
    def legs(self) -> int:
        return 0

for animal in [Dog(), Snake()]:
    print(f"  {animal.describe()}")

### 5.4 Abstract Methods with a Body — `super()` Trick

**☕ JAVA:** Abstract methods have **no body** — `public abstract double area();`

**🐍 PYTHON:** Abstract methods **can have a body**! Subclasses can call it via `super()` for shared default behavior. This is a powerful pattern for providing a base implementation that subclasses extend.

In [ ]:
from abc import ABC, abstractmethod
import json


class Formatter(ABC):
    def _clean(self, data: dict) -> dict:
        return {k: v for k, v in data.items() if v is not None}

    @abstractmethod
    def format(self, data: dict) -> str:
        return str(self._clean(data))


class JsonFormatter(Formatter):
    def format(self, data: dict) -> str:
        base = super().format(data)        # ✅ Can still call super()
        cleaned = self._clean(data)        # Get dict directly, no eval()
        return json.dumps(cleaned, indent=2)


class SimpleFormatter(Formatter):
    def format(self, data: dict) -> str:
        # Skip parent's logic entirely — that's also fine
        return ", ".join(f"{k}={v}" for k, v in data.items())


data = {"name": "Alice", "age": 30, "job": None}

print("JSON (uses super):")
print(f"  {JsonFormatter().format(data)}")
print(f"\nSimple (skips super):")
print(f"  {SimpleFormatter().format(data)}")

> 💡 **Key insight:** This is a Java/Python difference that surprises many developers. In Java, abstract methods cannot have a body at all. In Python, the `@abstractmethod` only enforces that subclasses *define* the method — the abstract method's body is available via `super()`.

### 5.5 `__init__` in Abstract Classes — Common Gotcha

**☕ JAVA:** Abstract classes commonly have constructors that subclasses call via `super()`.

**🐍 PYTHON:** Same pattern works! But note: **don't make `__init__` abstract**. Instead, define a concrete `__init__` in the ABC and have subclasses call `super().__init__()`.

In [ ]:
from datetime import datetime

class Entity(ABC):
    """Base class with shared constructor logic."""

    def __init__(self, entity_id: int):
        self.entity_id = entity_id
        self.created_at = datetime.now()

    @abstractmethod
    def validate(self) -> bool: ...

    def __repr__(self) -> str:
        return f"{self.__class__.__name__}(id={self.entity_id})"

class User(Entity):
    def __init__(self, entity_id: int, name: str):
        super().__init__(entity_id)   # Call ABC's __init__
        self.name = name

    def validate(self) -> bool:
        return len(self.name) > 0

class Product(Entity):
    def __init__(self, entity_id: int, price: float):
        super().__init__(entity_id)
        self.price = price

    def validate(self) -> bool:
        return self.price > 0

user = User(1, "Alice")
product = Product(2, 29.99)
print(f"{user}: valid={user.validate()}, created={user.created_at:%H:%M:%S}")
print(f"{product}: valid={product.validate()}, created={product.created_at:%H:%M:%S}")

> ⚠️ **Why not `@abstractmethod` on `__init__`?** Technically possible, but pointless — every class already needs `__init__` to be useful. The ABC is better off providing a concrete `__init__` with shared setup, and letting subclasses extend it via `super().__init__()`.

### 5.6 `ABC` vs `ABCMeta` — Under the Hood

`ABC` is just a convenience class. Under the hood, it uses `ABCMeta` as its metaclass:

```python
class ABC(metaclass=ABCMeta):   # This is all ABC does!
    pass
```

You only need `ABCMeta` directly when your class already has a different metaclass (rare).

In [ ]:
from abc import ABCMeta

# These two are equivalent:

# Approach 1: Simple (preferred)
class ShapeV1(ABC):
    @abstractmethod
    def area(self) -> float: ...

# Approach 2: Explicit metaclass (needed only for metaclass conflicts)
class ShapeV2(metaclass=ABCMeta):
    @abstractmethod
    def area(self) -> float: ...

print(f"ShapeV1 metaclass: {type(ShapeV1).__name__}")  # ABCMeta
print(f"ShapeV2 metaclass: {type(ShapeV2).__name__}")  # ABCMeta
print(f"Same thing: {type(ShapeV1) == type(ShapeV2)}")  # True

> 💡 **Rule of thumb:** Always use `class MyABC(ABC):` unless you get a metaclass conflict error — then switch to `class MyABC(OtherBase, metaclass=ABCMeta):`.

### 5.7 ABC with `__init_subclass__` — Registration Pattern

**☕ JAVA:** No direct equivalent — you'd use a registry `Map<String, Class>`.

**🐍 PYTHON:** `__init_subclass__` is called automatically when a class is subclassed. Great for plugin systems or automatic registration.

In [ ]:
class Plugin(ABC):
    """Auto-registers all subclasses."""
    registry: dict[str, type] = {}

    def __init_subclass__(cls, **kwargs):
        super().__init_subclass__(**kwargs)
        Plugin.registry[cls.__name__] = cls
        print(f"  Registered plugin: {cls.__name__}")

    @abstractmethod
    def execute(self) -> str: ...

# Subclassing automatically registers!
class EmailPlugin(Plugin):
    def execute(self) -> str:
        return "Sending email..."

class SlackPlugin(Plugin):
    def execute(self) -> str:
        return "Posting to Slack..."

print(f"\nAll plugins: {list(Plugin.registry.keys())}")

# Factory from registry
for name, cls in Plugin.registry.items():
    plugin = cls()
    print(f"  {name}: {plugin.execute()}")

### 5.8 ABC vs Protocol — When to Use Which?

| Feature | ABC | Protocol |
|---------|-----|----------|
| Style | Nominal (must inherit) | Structural (duck typing) |
| Java equivalent | `abstract class` / `interface` | No equivalent |
| Must inherit? | ✅ Yes | ❌ No — just implement the methods |
| Runtime check | `isinstance()` works | Only with `@runtime_checkable` |
| Enforcement | At instantiation time | At type-checking time (mypy) |
| Best for | Frameworks, plugins | Loose coupling, libraries |

In [ ]:
from typing import Protocol

# ABC approach — must inherit
class DrawableABC(ABC):
    @abstractmethod
    def draw(self) -> str: ...

# Protocol approach — just implement the method
class DrawableProtocol(Protocol):
    def draw(self) -> str: ...

# This class satisfies the Protocol WITHOUT inheriting from it
class Square:
    def draw(self) -> str:
        return "Drawing a square"

def render(obj: DrawableProtocol) -> None:
    print(f"  {obj.draw()}")

render(Square())   # ✅ Works — Square has draw(), that's enough!

---

## 🧪 Try It Yourself

**Exercise 1:** Create an abstract `Database` class with abstract methods `connect()`, `disconnect()`, and `query(sql)`. Implement `SQLiteDB` and `PostgresDB` that print simulated behavior.

In [ ]:
# Exercise 1: Your code here


**Exercise 2:** Create an abstract `Validator` with an abstract property `pattern` and a concrete method `validate(text)` that checks if text matches the pattern using `re.match()`. Implement `EmailValidator` and `PhoneValidator`.

In [ ]:
# Exercise 2: Your code here


**Exercise 3:** Create an abstract `PaymentProcessor` with `__init_subclass__` auto-registration. Implement `CreditCardProcessor` and `PayPalProcessor`. Write a factory function that creates a processor by name from the registry.

In [ ]:
# Exercise 3: Your code here


---

## 📝 Key Takeaways: Java → Python

| Concept | Java | Python |
|---------|------|--------|
| Abstract class | `abstract class Shape` | `class Shape(ABC):` |
| Abstract method | `public abstract double area();` | `@abstractmethod` |
| Import | Built-in keyword | `from abc import ABC, abstractmethod` |
| Abstract method body | ❌ Not allowed | ✅ Can have body, callable via `super()` |
| Concrete method in ABC | ✅ Same | ✅ Same |
| Abstract property | Abstract getter | `@property` + `@abstractmethod` |
| Abstract `__init__` | Common pattern | ❌ Don't — use concrete `__init__` instead |
| `ABC` vs `ABCMeta` | N/A | `ABC` = convenience; `ABCMeta` = for metaclass conflicts |
| Instantiation check | Compile-time error | Runtime `TypeError` |
| Missing method check | Compile-time error | Runtime `TypeError` at instantiation |
| Plugin registry | Manual `Map<String, Class>` | `__init_subclass__` auto-registration |
| Duck typing interface | Not possible | `Protocol` (structural typing) |
| Best practice | Use `abstract class` or `interface` | Prefer Protocol unless you need enforcement |